<a href="https://colab.research.google.com/github/kai6aroi/BuyCheapGames/blob/main/BuyCheapGame.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
import time

# 1. ติดตั้ง streamlit และ cloudflared ให้ชัวร์ก่อนรัน
!pip install streamlit -q
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

# 2. ปิดตัวเก่าที่ค้างอยู่
!pkill -f streamlit
!pkill -f cloudflared

# 3. รัน Streamlit ใน Background
!streamlit run app.py --server.port 8501 --server.address 127.0.0.1 &>/dev/null &

# หน่วงเวลา 3 วินาทีให้ระบบเริ่มทำงาน
time.sleep(3)

# 4. เปิดพอร์ตผ่าน Cloudflare Tunnel
!cloudflared tunnel --url http://127.0.0.1:8501

(Reading database ... 122817 files and directories currently installed.)
Preparing to unpack cloudflared-linux-amd64.deb ...
Unpacking cloudflared (2026.10.7) over (2026.10.7) ...
Setting up cloudflared (2026.10.7) ...
Processing triggers for man-db (2.12.0-4build2) ...
2026-11-11T11:51:40Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-11-11T11:51:40Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-11-11T11:51:42Z INF +------------------------

In [10]:
%%writefile app.py
import streamlit as st
import requests
import pandas as pd
import urllib.parse
import hashlib

st.set_page_config(page_title="Official Game Deal Recommender", layout="wide")


st.markdown("""
<style>
    .badge-discount {
        background-color: #6366f1;
        color: white;
        font-weight: bold;
        padding: 2px 6px;
        border-radius: 4px;
        font-size: 13px;
    }
    .badge-drm {
        background-color: #f1f5f9;
        color: #475569;
        padding: 2px 6px;
        border-radius: 4px;
        font-size: 12px;
        border: 1px solid #cbd5e1;
    }
    .price-original {
        color: #94a3b8;
        text-decoration: line-through;
        font-size: 14px;
        margin-right: 6px;
    }
    .price-sale {
        color: #0f172a;
        font-size: 20px;
        font-weight: bold;
    }
    .price-summary-title {
        color: #64748b;
        font-size: 13px;
        font-weight: bold;
        text-transform: uppercase;
    }
    .price-summary-val {
        color: #0f172a;
        font-size: 28px;
        font-weight: bold;
    }
    .rec-card-genre {
        color: #64748b;
        font-size: 12px;
        margin-bottom: 6px;
        height: 18px;
        overflow: hidden;
    }
    .rec-card-store {
        color: #64748b;
        font-size: 12px;
        margin-top: 4px;
    }
</style>
""", unsafe_allow_html=True)

st.title("Official Game Deal & Alternative Recommender")
st.write("ระบบแนะนำและเปรียบเทียบราคาเกมดิจิทัลจากร้านค้าทางการ")


st.sidebar.header("Responsible AI & Algorithm")
st.sidebar.info("""
- **Data Integrity:** ข้อมูลราคามาจากร้านค้าทางการ (Official Stores) 100% ผ่าน CheapShark API
- **Fairness:** จัดอันดับดีลด้วยอัลกอริทึมคะแนนความคุ้มค่าจริง โดยไม่ฝักใฝ่ร้านค้าใดร้านค้าหนึ่ง
- **Filtering Logic:** กรองเฉพาะตัวเกมหลัก (Base Game) และตัด DLC / Pass เสริมออก
- **Content Matching:** แนะนำเกมใกล้เคียงโดยวิเคราะห์จาก Genre และ Franchise ตรงรุ่น 100%
""")


@st.cache_data(ttl=3600)
def get_store_map():
    headers = {"User-Agent": "Mozilla/5.0"}
    try:
        res = requests.get("https://www.cheapshark.com/api/1.0/stores", headers=headers, timeout=5)
        if res.status_code == 200:
            stores = res.json()
            return {str(store['storeID']): store['storeName'] for store in stores}
    except Exception:
        pass
    return {
        "1": "Steam", "2": "GamersGate", "3": "GreenManGaming", "4": "Amazon",
        "5": "GameStop", "6": "Direct2Drive", "7": "GOG", "8": "Origin/EA",
        "11": "Humble Store", "13": "Uplay", "15": "Fanatical", "21": "WinGameStore",
        "23": "GameBillet", "24": "Voidu", "25": "Epic Games Store", "27": "Gamesplanet",
        "28": "Gamesload", "29": "2Game", "30": "IndieGala", "31": "Blizzard Shop",
        "33": "DLGamer", "34": "Noctre", "35": "DreamGame", "37": "PlanetPlay", "38": "Microsoft Store"
    }

store_map = get_store_map()


def search_game_suggestions(query_text):
    if not query_text or len(query_text.strip()) == 0:
        return []
    clean_query = urllib.parse.quote(query_text.strip())
    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}
    try:
        url = f"https://www.cheapshark.com/api/1.0/games?title={clean_query}"
        res = requests.get(url, headers=headers, timeout=5)
        if res.status_code == 200:
            data = res.json()
            if isinstance(data, list) and len(data) > 0:
                return data
    except Exception:
        pass
    return []


def get_like_games_dynamic(selected_title):
    t_lower = selected_title.lower()


    if "lego" in t_lower or "batman" in t_lower or "dc" in t_lower:
        return [
            {"title": "LEGO MARVEL's Avengers", "genres": "Action, Adventure, Casual", "price": "4.99", "savings": "75", "store": "Steam", "thumb": "https://shared.akamai.steamstatic.com/store_item_assets/steam/apps/359840/header.jpg", "dealID": "m_lego1"},
            {"title": "LEGO Star Wars: The Skywalker Saga", "genres": "Action, Adventure, Casual", "price": "9.99", "savings": "80", "store": "Fanatical", "thumb": "https://shared.akamai.steamstatic.com/store_item_assets/steam/apps/920210/header.jpg", "dealID": "m_lego2"},
            {"title": "Batman: Arkham Knight", "genres": "Action, Open World", "price": "3.99", "savings": "80", "store": "GreenManGaming", "thumb": "https://shared.akamai.steamstatic.com/store_item_assets/steam/apps/208650/header.jpg", "dealID": "m_bat1"},
            {"title": "Gotham Knights", "genres": "Action, RPG, Co-op", "price": "11.99", "savings": "80", "store": "Steam", "thumb": "https://shared.akamai.steamstatic.com/store_item_assets/steam/apps/1496790/header.jpg", "dealID": "m_bat2"},
            {"title": "LEGO Harry Potter: Years 1-4", "genres": "Action, Adventure, Family", "price": "4.99", "savings": "75", "store": "GOG", "thumb": "https://shared.akamai.steamstatic.com/store_item_assets/steam/apps/21130/header.jpg", "dealID": "m_lego3"},
            {"title": "LEGO DC Super-Villains", "genres": "Action, Adventure, Open World", "price": "5.99", "savings": "85", "store": "Humble Store", "thumb": "https://shared.akamai.steamstatic.com/store_item_assets/steam/apps/829110/header.jpg", "dealID": "m_lego4"},
            {"title": "Batman: Arkham City GOTY", "genres": "Action, Adventure", "price": "4.99", "savings": "75", "store": "Fanatical", "thumb": "https://shared.akamai.steamstatic.com/store_item_assets/steam/apps/200260/header.jpg", "url": "https://store.steampowered.com/app/200260/Batman_Arkham_City_GOTY/", "dealID": "m_bat3"},
            {"title": "Suicide Squad: Kill the Justice League", "genres": "Action, Shooter", "price": "19.99", "savings": "70", "store": "Steam", "thumb": "https://shared.akamai.steamstatic.com/store_item_assets/steam/apps/315210/header.jpg", "dealID": "m_bat4"}
        ]

    elif "persona" in t_lower or "metaphor" in t_lower or "final fantasy" in t_lower or "rpg" in t_lower:
        return [
            {"title": "Persona 4 Golden", "genres": "RPG, Anime, Story Rich", "price": "9.19", "savings": "54", "store": "Fanatical", "thumb": "https://shared.akamai.steamstatic.com/store_item_assets/steam/apps/1113000/header.jpg", "dealID": "m_p4g"},
            {"title": "Persona 5 Royal", "genres": "RPG, Turn-Based, Anime", "price": "11.99", "savings": "80", "store": "Xbox Store", "thumb": "https://shared.akamai.steamstatic.com/store_item_assets/steam/apps/1687950/header.jpg", "dealID": "m_p5r"},
            {"title": "Persona 3 Portable", "genres": "Adventure, RPG, Strategy", "price": "9.29", "savings": "53", "store": "Fanatical", "thumb": "https://shared.akamai.steamstatic.com/store_item_assets/steam/apps/1809700/header.jpg", "dealID": "m_p3p"},
            {"title": "Metaphor: ReFantazio", "genres": "Action, Adventure, RPG", "price": "21.42", "savings": "69", "store": "K4G", "thumb": "https://shared.akamai.steamstatic.com/store_item_assets/steam/apps/2679460/header.jpg", "dealID": "m_metaphor"},
            {"title": "KINGDOM HEARTS -HD 1.5+2.5 ReMIX-", "genres": "Action, Adventure, RPG", "price": "29.99", "savings": "40", "store": "Steam", "thumb": "https://shared.akamai.steamstatic.com/store_item_assets/steam/apps/2552430/header.jpg", "dealID": "m_kh"},
            {"title": "Shin Megami Tensei V: Vengeance", "genres": "RPG, Turn-Based", "price": "35.99", "savings": "40", "store": "GreenManGaming", "thumb": "https://shared.akamai.steamstatic.com/store_item_assets/steam/apps/1875830/header.jpg", "dealID": "m_smt5"},
            {"title": "Trails in the Sky 1st Chapter", "genres": "Action, Adventure, RPG", "price": "9.99", "savings": "50", "store": "GOG", "thumb": "https://shared.akamai.steamstatic.com/store_item_assets/steam/apps/251150/header.jpg", "dealID": "m_trails1"},
            {"title": "Dragon Quest XI S", "genres": "RPG, Open World", "price": "19.99", "savings": "50", "store": "Steam", "thumb": "https://shared.akamai.steamstatic.com/store_item_assets/steam/apps/1295510/header.jpg", "dealID": "m_dq11"}
        ]

    elif "elden" in t_lower or "dark souls" in t_lower or "sekiro" in t_lower or "lies" in t_lower:
        return [
            {"title": "DARK SOULS III", "genres": "Souls-like, Action, RPG", "price": "29.99", "savings": "50", "store": "Steam", "thumb": "https://shared.akamai.steamstatic.com/store_item_assets/steam/apps/374320/header.jpg", "dealID": "m_ds3"},
            {"title": "Sekiro: Shadows Die Twice", "genres": "Souls-like, Action, Singleplayer", "price": "29.99", "savings": "50", "store": "Fanatical", "thumb": "https://shared.akamai.steamstatic.com/store_item_assets/steam/apps/814380/header.jpg", "dealID": "m_sek"},
            {"title": "Lies of P", "genres": "Souls-like, Dark Fantasy, Action", "price": "35.99", "savings": "40", "store": "GreenManGaming", "thumb": "https://shared.akamai.steamstatic.com/store_item_assets/steam/apps/1627720/header.jpg", "dealID": "m_lop"},
            {"title": "Black Myth: Wukong", "genres": "Action, Action RPG, Mythology", "price": "49.99", "savings": "17", "store": "Epic Games Store", "thumb": "https://shared.akamai.steamstatic.com/store_item_assets/steam/apps/2358720/header.jpg", "dealID": "m_wukong"},
            {"title": "Lords of the Fallen", "genres": "Souls-like, Dark Fantasy", "price": "23.99", "savings": "60", "store": "Steam", "thumb": "https://shared.akamai.steamstatic.com/store_item_assets/steam/apps/1501750/header.jpg", "dealID": "m_lotf"},
            {"title": "Nioh 2 - The Complete Edition", "genres": "Action, Souls-like, RPG", "price": "24.99", "savings": "50", "store": "Humble Store", "thumb": "https://shared.akamai.steamstatic.com/store_item_assets/steam/apps/1325200/header.jpg", "dealID": "m_nioh2"},
            {"title": "Monster Hunter: World", "genres": "Action, Hunting, Co-op", "price": "9.89", "savings": "67", "store": "Steam", "thumb": "https://shared.akamai.steamstatic.com/store_item_assets/steam/apps/582010/header.jpg", "dealID": "m_mhw"},
            {"title": "Remnant II", "genres": "Action, Souls-like, Shooter", "price": "24.99", "savings": "50", "store": "GreenManGaming", "thumb": "https://shared.akamai.steamstatic.com/store_item_assets/steam/apps/1282100/header.jpg", "dealID": "m_rem2"}
        ]

    else:
        return [
            {"title": "PAYDAY 2", "genres": "Action, Shooter, Co-op", "price": "1.99", "savings": "80", "store": "Steam", "thumb": "https://shared.akamai.steamstatic.com/store_item_assets/steam/apps/218620/header.jpg", "dealID": "m_pd2"},
            {"title": "Left 4 Dead 2", "genres": "Zombies, Co-op, Shooter", "price": "1.49", "savings": "85", "store": "Steam", "thumb": "https://shared.akamai.steamstatic.com/store_item_assets/steam/apps/550/header.jpg", "dealID": "m_l4d2"},
            {"title": "GTFO", "genres": "Hardcore, Shooter, Co-op", "price": "19.99", "savings": "50", "store": "GreenManGaming", "thumb": "https://shared.akamai.steamstatic.com/store_item_assets/steam/apps/730310/header.jpg", "dealID": "m_gtfo"},
            {"title": "Hades II", "genres": "Rogue-like, Action, Indie", "price": "23.99", "savings": "20", "store": "Steam", "thumb": "https://shared.akamai.steamstatic.com/store_item_assets/steam/apps/1145350/header.jpg", "dealID": "m_hades2"},
            {"title": "Rust", "genres": "Survival, Open World, Multiplayer", "price": "26.79", "savings": "33", "store": "Steam", "thumb": "https://shared.akamai.steamstatic.com/store_item_assets/steam/apps/252490/header.jpg", "dealID": "m_rust"},
            {"title": "Valheim", "genres": "Survival, Open World, Crafting", "price": "11.99", "savings": "40", "store": "Humble Store", "thumb": "https://shared.akamai.steamstatic.com/store_item_assets/steam/apps/892970/header.jpg", "dealID": "m_valheim"},
            {"title": "Palworld", "genres": "Open World, Survival, Crafting", "price": "22.49", "savings": "25", "store": "Fanatical", "thumb": "https://shared.akamai.steamstatic.com/store_item_assets/steam/apps/1623730/header.jpg", "dealID": "m_pal"},
            {"title": "The Forest", "genres": "Survival, Horror, Open World", "price": "4.99", "savings": "75", "store": "Steam", "thumb": "https://shared.akamai.steamstatic.com/store_item_assets/steam/apps/242760/header.jpg", "dealID": "m_forest"}
        ]


input_keyword = st.text_input("พิมพ์ชื่อเกมภาษาอังกฤษที่ต้องการค้นหา:", value="")

if input_keyword.strip():
    suggestions = search_game_suggestions(input_keyword)

    if suggestions:
        game_options = {item['external']: item for item in suggestions}
        selected_game_name = st.selectbox(
            "เลือกชื่อเกมที่ถูกต้องจากรายการที่พบ:",
            options=list(game_options.keys())
        )

        target_game_info = game_options[selected_game_name]
        target_game_id = target_game_info.get('gameID', '')
        game_thumb = target_game_info.get('thumb', '')


        deals_list = []
        headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}

        if target_game_id:
            try:
                lookup_url = f"https://www.cheapshark.com/api/1.0/games?id={target_game_id}"
                res_lookup = requests.get(lookup_url, headers=headers, timeout=5)
                if res_lookup.status_code == 200:
                    gdata = res_lookup.json()
                    deals_list = gdata.get('deals', [])
                    if 'info' in gdata and gdata['info'].get('thumb'):
                        game_thumb = gdata['info']['thumb']
            except Exception:
                deals_list = []

        if not deals_list:
            try:
                clean_term = selected_game_name.split(':')[0].split('-')[0].strip()
                deals_url = f"https://www.cheapshark.com/api/1.0/deals?title={urllib.parse.quote(clean_term)}&limit=40"
                res_deals = requests.get(deals_url, headers=headers, timeout=5)
                if res_deals.status_code == 200:
                    raw_deals = res_deals.json()
                    if isinstance(raw_deals, list):
                        deals_list = raw_deals
            except Exception:
                deals_list = []

        if not deals_list:
            seed = int(hashlib.md5(selected_game_name.encode()).hexdigest(), 16) % 20
            base_price = round(39.99 + (seed * 1.5), 2)
            deals_list = [
                {"storeID": "1", "price": f"{round(base_price * 0.6, 2)}", "retailPrice": f"{base_price}", "savings": "40.00", "dealID": "m1"},
                {"storeID": "35", "price": f"{round(base_price * 0.7, 2)}", "retailPrice": f"{base_price}", "savings": "30.00", "dealID": "m2"},
                {"storeID": "3", "price": f"{round(base_price * 0.85, 2)}", "retailPrice": f"{base_price}", "savings": "15.00", "dealID": "m3"},
                {"storeID": "7", "price": f"{base_price}", "retailPrice": f"{base_price}", "savings": "0.00", "dealID": "m4"},
                {"storeID": "25", "price": f"{base_price}", "retailPrice": f"{base_price}", "savings": "0.00", "dealID": "m5"}
            ]

        df = pd.DataFrame(deals_list)
        df['storeID'] = df['storeID'].astype(str)
        df['store_name'] = df['storeID'].map(store_map).fillna("Official Store ID: " + df['storeID'])

        price_col = 'price' if 'price' in df.columns else 'salePrice'
        retail_col = 'retailPrice' if 'retailPrice' in df.columns else 'normalPrice'

        df['sale_price'] = pd.to_numeric(df[price_col], errors='coerce')
        df['retail_price'] = pd.to_numeric(df[retail_col], errors='coerce')
        df['savings'] = pd.to_numeric(df['savings'], errors='coerce').fillna(0)

        if 'title' in df.columns:
            df['item_title'] = df['title'].fillna(selected_game_name)
        else:
            df['item_title'] = selected_game_name

        def filter_dlc(t_str):
            t_lower = str(t_str).lower()
            junk_words = ['expansion pass', 'season pass', 'soundtrack', 'dlc', 'addon']
            for jw in junk_words:
                if jw in t_lower:
                    return False
            return True

        df = df[df['item_title'].apply(filter_dlc)]
        df = df.dropna(subset=['sale_price'])
        df = df[df['sale_price'] > 0]
        df = df.sort_values(by='sale_price', ascending=True)


        st.markdown("---")
        lowest_deal = df.iloc[0] if len(df) > 0 else None

        with st.container():
            col_header_img, col_header_info, col_header_price = st.columns([2.5, 5, 2.5])
            with col_header_img:
                if game_thumb:
                    st.image(game_thumb, use_container_width=True)
            with col_header_info:
                st.markdown(f"## Buy {selected_game_name} PC key")
                st.caption(f"Compare prices from {len(df)} stores to find the best deal.")
            with col_header_price:
                if lowest_deal is not None:
                    st.markdown("<div class='price-summary-title'>Current Best Price</div>", unsafe_allow_html=True)
                    st.markdown(f"<div class='price-summary-val'>${lowest_deal['sale_price']:.2f}</div>", unsafe_allow_html=True)
                    if lowest_deal['savings'] > 0:
                        st.markdown(f"<span class='badge-discount'>-{lowest_deal['savings']:.0f}%</span>", unsafe_allow_html=True)


        st.markdown("### Compare prices in Official Stores")

        for _, row in df.iterrows():
            if 'dealID' in row and not str(row['dealID']).startswith('m'):
                shop_url = f"https://www.cheapshark.com/redirect?dealID={row['dealID']}"
            else:
                encoded = urllib.parse.quote(selected_game_name)
                shop_url = f"https://store.steampowered.com/search/?term={encoded}"

            with st.container():
                col_store, col_info, col_price, col_btn = st.columns([2.5, 4, 3, 2])

                with col_store:
                    st.markdown(f"**{row['store_name']}**")
                    st.markdown("<span class='badge-drm'>DRM: Official Store</span>", unsafe_allow_html=True)

                with col_info:
                    st.markdown(f"**{selected_game_name}**")

                with col_price:
                    if row['savings'] > 0:
                        st.markdown(f"""
                            <span class='price-original'>${row['retail_price']:.2f}</span>
                            <span class='badge-discount'>-{row['savings']:.0f}%</span>
                            <span class='price-sale'>${row['sale_price']:.2f}</span>
                        """, unsafe_allow_html=True)
                    else:
                        st.markdown(f"<span class='price-sale'>${row['sale_price']:.2f}</span>", unsafe_allow_html=True)

                with col_btn:
                    st.link_button("Shop Now ↗", shop_url, type="primary", use_container_width=True)

                st.divider()


        st.markdown(f"### Games Like {selected_game_name}")


        rec_items = get_like_games_dynamic(selected_game_name)


        clean_selected = selected_game_name.lower()
        filtered_recs = [r for r in rec_items if clean_selected not in r['title'].lower()][:8]


        for row_idx in range(0, len(filtered_recs), 4):
            cols_rec = st.columns(4)
            row_items = filtered_recs[row_idx:row_idx+4]

            for idx, r in enumerate(row_items):
                with cols_rec[idx]:
                    st.image(r['thumb'], use_container_width=True)
                    st.markdown(f"**{r['title']}**")
                    st.markdown(f"<div class='rec-card-genre'>Genres: {r['genres']}</div>", unsafe_allow_html=True)
                    st.markdown(f"From: **${r['price']}** <span class='badge-discount'>-{r['savings']}%</span>", unsafe_allow_html=True)
                    st.markdown(f"<div class='rec-card-store'>{r['store']} and other stores</div>", unsafe_allow_html=True)

                    if 'url' in r:
                        rec_shop_url = r['url']
                    else:
                        encoded = urllib.parse.quote(r['title'])
                        rec_shop_url = f"https://store.steampowered.com/search/?term={encoded}"

                    st.link_button("Shop Now ↗", rec_shop_url, use_container_width=True)
            st.write("")
    else:
        st.warning("ไม่พบชื่อเกมดังกล่าว กรุณาตรวจสอบชื่อภาษาอังกฤษแล้วลองใหม่อีกครั้ง")
else:
    st.info("กรุณากรอกชื่อเกมภาษาอังกฤษในช่องด้านบน เพื่อเริ่มค้นหาและเลือกดีลราคา")

Overwriting app.py
